## Data preprocessing

### 0 - Sagemaker config

In [ ]:
import pandas as pd
import os

import boto3
import sagemaker

import tensorflow as tf
from tensorflow.keras.preprocessing import sequence

In [ ]:
# region identificaion
region = sagemaker.Session().boto_region_name
print("AWS Region: {}".format(region))

# role identification
role = sagemaker.get_execution_role()
print("RoleArn: {}".format(role))

# create sesion
sagemaker_session = sagemaker.Session()

### 1 - Get data and exploration

**Url data**

https://www.tensorflow.org/api_docs/python/tf/keras/datasets/imdb/load_data

In [ ]:
# max number of vocavulary, return the n word most used
max_features = 20000

# review's max lenght 
maxlen = 500

# download data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=max_features)

print(len(x_train), "train sequences")
print(len(x_test), "test sequences")

**Data exploration**

In [ ]:
print(f"Maximum word index in the dataset: {max(word for review in x_train for word in review)}")

In [ ]:
print(f"example: {x_train[0]}")

In [ ]:
# words to index
word_to_integer = tf.keras.datasets.imdb.get_word_index()

# example of common word
print(list(word_to_integer.items())[0:5])

In [ ]:
# example
word_to_integer['the']

In [ ]:
# word to index
integer_to_word = dict([(value, key) for (key, value) in word_to_integer.items()])
#
#  example
integer_to_word[1]

In [ ]:
# We need to subtract 3 from each index:
# 0 is reserved for padding, 1 for the start of sequence, and 2 for unknown words.
# Each review has a different length.

decoded_review = ' '.join([integer_to_word.get(i - 3, 'UNK') for i in x_train[0]])
print(decoded_review)

### 2 - Padding

In [ ]:
# same lenght
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

### 3 - Save data

In [ ]:
df_x_train = pd.DataFrame(x_train)
df_y_train = pd.DataFrame(y_train)
df_x_test = pd.DataFrame(x_test)
df_y_test = pd.DataFrame(y_test)

In [ ]:
df_train = pd.concat([df_y_train, df_x_train], axis=1)
df_test_and_validation = pd.concat([df_y_test, df_x_test], axis=1)

In [ ]:
df_validation = df_test_and_validation[:20000]
df_test = df_test_and_validation[20000:]

**Local**

In [ ]:
# Create dir
data_dir_train = "./data/pd/train"
if not os.path.exists(data_dir_train):
    os.makedirs(data_dir_train)

data_dir_test = "./data/pd/test"
if not os.path.exists(data_dir_test):
    os.makedirs(data_dir_test)

df_train.to_csv(os.path.join(data_dir_train, 'train.csv'), header=False, index=False)
df_validation.to_csv(os.path.join(data_dir_train, 'validation.csv'), header=False, index=False)
df_test.to_csv(os.path.join(data_dir_test, 'test.csv'), header=False, index=False)

**En S3**

In [ ]:
# S3 base path where the dataset will be stored
s3_folder_bucket = "clasificador_sentimiento/data"

# Upload the training dataset to S3 under the "train" prefix
train_s3 = sagemaker_session.upload_data(
    os.path.join(data_dir_train, 'train.csv'),
    key_prefix=f"{s3_folder_bucket}/train"
)

# Upload the validation dataset to S3 (same prefix as train, but different file)
validation_s3 = sagemaker_session.upload_data(
    os.path.join(data_dir_train, 'validation.csv'),
    key_prefix=f"{s3_folder_bucket}/train"
)

# Upload the test dataset to S3 under the "test" prefix
test_s3 = sagemaker_session.upload_data(
    os.path.join(data_dir_test, 'test.csv'),
    key_prefix=f"{s3_folder_bucket}/test"
)

# Dictionary containing the S3 paths for each dataset split
inputs = {
    "train": train_s3,
    "validation": validation_s3,
    "test": test_s3
}

inputs